# Regular-grid Reverse-L Column Vecchia test

This notebook fits the new `ReverseLColumnVecchiaFit` kernel on real GEMS TCO data using regular grid coordinates only (`keep_ori=False`).

Conditioning logic:

- rightmost 3 grid columns are exact head points;
- all later columns are scanned right-to-left, north-to-south;
- each tail target uses 4 above points plus about 80 points from the next 3 east/right columns;
- the same spatial stencil is repeated at t, t-1, and t-2;
- repeated offset patterns are cached as templates inside the kernel.


In [1]:
import sys, time, gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch

sys.path.insert(0, "/Users/joonwonlee/Documents/GEMS_TCO-1/src")
from GEMS_TCO import configuration as config
from GEMS_TCO.data_loader import load_data_dynamic_processed
from GEMS_TCO.kernels_vecchia_column import ReverseLColumnVecchiaFit

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())


torch: 2.5.1
cuda available: False


In [2]:
# Experiment config
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE = torch.float64

YEAR = '2024'
MONTH = 7
DAY_IDX_LIST = [2]  # 0-based; [2] = July 3. Change to list(range(5)) for 5 days.
LAT_RANGE = [-3, 2]
LON_RANGE = [121, 131]

MM_COND_NUMBER = 100  # only used for monthly maxmin ordering in the loader

# Reverse-L column stencil
HEAD_RIGHT_COLS = 3
ABOVE_COUNT = 4
RIGHT_COL_COUNT = 3
RIGHT_NEIGHBOR_COUNT = 80
LAG_COUNT = 2
INCLUDE_LAG_SELF = False
TARGET_CHUNK_SIZE = 1024 if DEVICE.type == 'cpu' else 4096

# Fitting controls
FIT_SMOOTHS = [0.5]  # add 1.5 after smoke test if desired
LBFGS_LR = 1.0
LBFGS_STEPS = 5
LBFGS_EVAL = 15
LBFGS_HIST = 10

INIT_DICT = {
    'sigmasq':    13.059,
    'range_lat':  0.2,
    'range_lon':  0.25,
    'range_time': 1.5,
    'advec_lat':  0.0218,
    'advec_lon': -0.1689,
    'nugget':     0.247,
}
P_LABELS = ['sigmasq', 'range_lat', 'range_lon', 'range_time', 'advec_lat', 'advec_lon', 'nugget']

OUT_DIR = Path('/Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/testing/log')
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = OUT_DIR / 'real_reverse_l_column_vecchia_050526_results.csv'

print('device:', DEVICE)
print('days:', DAY_IDX_LIST)
print('output:', OUT_CSV)


device: cpu
days: [2]
output: /Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/testing/log/real_reverse_l_column_vecchia_050526_results.csv


In [3]:
def phys_to_log(d):
    phi2 = 1.0 / d['range_lon']
    phi1 = d['sigmasq'] * phi2
    phi3 = (d['range_lon'] / d['range_lat']) ** 2
    phi4 = (d['range_lon'] / d['range_time']) ** 2
    return [np.log(phi1), np.log(phi2), np.log(phi3), np.log(phi4),
            d['advec_lat'], d['advec_lon'], np.log(d['nugget'])]


def backmap_params(out_params):
    p = [float(x.item() if isinstance(x, torch.Tensor) else x) for x in out_params[:7]]
    phi2 = np.exp(p[1])
    phi3 = np.exp(p[2])
    phi4 = np.exp(p[3])
    rlon = 1.0 / phi2
    return {
        'sigmasq':    np.exp(p[0]) / phi2,
        'range_lat':  rlon / phi3 ** 0.5,
        'range_lon':  rlon,
        'range_time': rlon / phi4 ** 0.5,
        'advec_lat':  p[4],
        'advec_lon':  p[5],
        'nugget':     np.exp(p[6]),
    }

INIT_LOG = phys_to_log(INIT_DICT)
print('Initial log params:', [round(x, 4) for x in INIT_LOG])


Initial log params: [3.9558, 1.3863, 0.4463, -3.5835, 0.0218, -0.1689, -1.3984]


In [4]:
# Load full month once.  We still request Vecchia maxmin ordering so rows match the existing pipeline,
# but the column kernel reconstructs regular grid row/column structure from Latitude/Longitude.
loader = load_data_dynamic_processed(config.mac_data_load_path)

df_map, ord_mm, nns_map_full, monthly_mean = loader.load_maxmin_ordered_data_bymonthyear(
    lat_lon_resolution=[1, 1],
    mm_cond_number=MM_COND_NUMBER,
    years_=[YEAR],
    months_=[MONTH],
    lat_range=LAT_RANGE,
    lon_range=LON_RANGE,
    is_whittle=False,
)

sorted_df_keys = sorted(df_map.keys())
first_df = df_map[sorted_df_keys[0]].iloc[ord_mm].reset_index(drop=True)
grid_coords_ordered = first_df[['Latitude', 'Longitude']].values.astype(np.float64)

print(f'Monthly mean TCO: {monthly_mean:.4f}')
print(f'Time slots: {len(sorted_df_keys)}')
print(f'Grid cells: {len(grid_coords_ordered):,}')
print('lat:', grid_coords_ordered[:,0].min(), grid_coords_ordered[:,0].max())
print('lon:', grid_coords_ordered[:,1].min(), grid_coords_ordered[:,1].max())


--- Global Monthly Mean for 2024-7: 257.9726 ---
--- Generating NNS Map for Vecchia (C++ Accelerated) ---
Monthly mean TCO: 257.9726
Time slots: 248
Grid cells: 18,126
lat: -2.9720000000000044 2.0
lon: 121.04600000000188 131.0


In [ ]:
def fit_one_day(DAY_IDX, smooth):
    date_str = f'{YEAR}{MONTH:02d}{DAY_IDX + 1:02d}'
    hour_indices = [DAY_IDX * 8, (DAY_IDX + 1) * 8]
    day_keys = sorted_df_keys[hour_indices[0]:hour_indices[1]]

    # IMPORTANT: keep_ori=False means Latitude/Longitude regular-grid coordinates are used.
    day_map, _ = loader.load_working_data(
        df_map, monthly_mean, hour_indices,
        ord_mm=ord_mm, dtype=DTYPE, keep_ori=False,
    )
    day_map = {k: v.to(DEVICE) for k, v in day_map.items()}
    n_valid = sum(int((~torch.isnan(v[:, 2])).sum().item()) for v in day_map.values())

    print('\n' + '=' * 80)
    print(f'DAY {date_str} | smooth={smooth} | {day_keys[0]} ... {day_keys[-1]} | valid={n_valid:,}')

    params = [
        torch.tensor([v], device=DEVICE, dtype=DTYPE, requires_grad=True)
        for v in INIT_LOG
    ]

    model = ReverseLColumnVecchiaFit(
        smooth=smooth,
        input_map=day_map,
        mm_cond_number=MM_COND_NUMBER,
        grid_coords=grid_coords_ordered,
        head_right_cols=HEAD_RIGHT_COLS,
        above_count=ABOVE_COUNT,
        right_col_count=RIGHT_COL_COUNT,
        right_neighbor_count=RIGHT_NEIGHBOR_COUNT,
        lag_count=LAG_COUNT,
        include_lag_self=INCLUDE_LAG_SELF,
        target_chunk_size=TARGET_CHUNK_SIZE,
    )
    model.precompute_conditioning_sets()

    optimizer = model.set_optimizer(
        params, lr=LBFGS_LR,
        max_iter=LBFGS_EVAL, max_eval=LBFGS_EVAL,
        history_size=LBFGS_HIST,
    )
    t0 = time.time()
    out, fit_iter = model.fit_vecc_lbfgs(params, optimizer, max_steps=LBFGS_STEPS, grad_tol=1e-5)
    elapsed = time.time() - t0

    loss = float(out[-1])
    est = backmap_params(out)
    row = {
        'date_str': date_str,
        'day_idx': DAY_IDX,
        'smooth': smooth,
        'loss': loss,
        'fit_steps': fit_iter + 1,
        'elapsed_s': round(elapsed, 2),
        'n_valid': n_valid,
        'n_heads': int(model.Heads_data.shape[0]),
        'n_tails': int(model.n_tails),
        'n_templates': int(len(model.Grouped_Batches)),
        'head_right_cols': HEAD_RIGHT_COLS,
        'above_count': ABOVE_COUNT,
        'right_col_count': RIGHT_COL_COUNT,
        'right_neighbor_count': RIGHT_NEIGHBOR_COUNT,
        'lag_count': LAG_COUNT,
        'include_lag_self': INCLUDE_LAG_SELF,
        **{f'est_{k}': est[k] for k in P_LABELS},
    }
    print('RESULT:', row)

    del model
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    return row


: 

In [ ]:
rows = []
for day_idx in DAY_IDX_LIST:
    for smooth in FIT_SMOOTHS:
        rows.append(fit_one_day(day_idx, smooth))
        pd.DataFrame(rows).to_csv(OUT_CSV, index=False)

df_results = pd.DataFrame(rows)
df_results



DAY 20240703 | smooth=0.5 | 2024_07_y24m07day03_hm00:53 ... 2024_07_y24m07day03_hm07:48 | valid=140,352
Pre-computing ReverseLColumnVecchia [heads_right=3, above=4, right_cols=3, right_n=80, lags=2]... Done in 31.6s. grid=114x159, heads=2509, tails=137843, templates=46018, m mean/med/max=213.4/246/252
--- Starting Reverse-L Column Vecchia L-BFGS ---


In [ ]:
print('Saved:', OUT_CSV)
if len(df_results):
    display_cols = ['date_str', 'smooth', 'loss', 'elapsed_s', 'n_heads', 'n_tails', 'n_templates',
                    'est_range_lat', 'est_range_lon', 'est_range_time', 'est_advec_lon', 'est_nugget']
    display(df_results[display_cols].sort_values(['date_str', 'smooth']))


## Notes

This is deliberately a regular-grid template approximation.  It does not use `Source_Latitude` / `Source_Longitude`.

The most important diagnostics after the first run are:

- `n_templates`: if this is small, template reuse is working; if it is huge, missing observations are fragmenting the templates.
- `elapsed_s`: compare against the existing `HybridVecchiaFit` runs.
- `loss` and estimates: compare cautiously, because this is a different conditioning approximation and uses grid coordinates exactly.
